In [0]:
from pyspark.sql.functions import col, lit, current_timestamp

print("Step 1: Required libraries imported successfully")

In [0]:
dbutils.widgets.text("landing_timestamp", "")
dbutils.widgets.text("table_name", "")
dbutils.widgets.text("merge_keys", "")
dbutils.widgets.text("load_type", "")

# variables
landing_timestamp = dbutils.widgets.get("landing_timestamp")
table_name = dbutils.widgets.get("table_name")
merge_keys = dbutils.widgets.get("merge_keys")
load_type = dbutils.widgets.get("load_type")

print("Step 2: Widgets created and parameters read")
print(f"  landing_timestamp : {landing_timestamp}")
print(f"  table_name        : {table_name}")
print(f"  merge_keys        : {merge_keys}")
print(f"  load_type         : {load_type}")

In [0]:
bronze_df = spark.read.table(f"adb_caresync.bronze.{table_name}").filter(f"landing_timestamp = '{landing_timestamp}'")

source_count = bronze_df.count()
print(f"Step 3: Read incremental data from adb_caresync.bronze.{table_name}")
print(f"  Records found for landing_timestamp '{landing_timestamp}': {source_count}")
if source_count == 0:
    print(f"  No records found for landing_timestamp '{landing_timestamp}'. Skipping silver load and exiting notebook.")
    dbutils.notebook.exit("0")

In [0]:
target_table = f"adb_caresync.silver.{table_name}"
normalized_load_type = (load_type or "Full").strip().upper()

merge_key_list = [key.strip() for key in merge_keys.split(",") if key.strip()]

if normalized_load_type == "MERGE":
    if not merge_key_list:
        raise ValueError("merge_keys must be provided when load_type is 'MERGE'.")

    missing_merge_keys = [key for key in merge_key_list if key not in bronze_df.columns]
    if missing_merge_keys:
        raise ValueError(f"merge_keys not found in source data : {missing_merge_keys}")

updated_bronze_df = bronze_df.withColumn("insert_timestamp", current_timestamp())\
                     .withColumn("update_timestamp", current_timestamp())

print("Step 4: Load metadata prepared")
print(f"  Target table : {target_table}")
print(f"  Load type    : {normalized_load_type}")
print(f"  Merge keys   : {merge_key_list if merge_key_list else 'N/A'}")
print(f"  Columns      : {updated_bronze_df.columns}")

In [0]:
print(f"Step 5: Writing data to {target_table} using {normalized_load_type} load")

if normalized_load_type == "FULL":
    updated_bronze_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(target_table)
    print(f"  FULL load completed: {target_table} overwritten")
elif normalized_load_type == "APPEND":
    updated_bronze_df.write.mode("append").option("mergeSchema", "true").saveAsTable(target_table)
    print(f"  APPEND load completed: records appended to {target_table}")
elif normalized_load_type == "MERGE":
    if not spark.catalog.tableExists(target_table):
        print(f"  Target table {target_table} does not exist, creating it")
        updated_bronze_df.write.mode("overwrite").saveAsTable(target_table)
        print(f"  MERGE load completed: {target_table} created")
    else:
        source_view = "silver_merge_source"
        updated_bronze_df.createOrReplaceTempView(source_view)

        merge_condition = " AND ".join(
            [f"target.`{key}` = source.`{key}`" for key in merge_key_list]
        )

        update_assignments = ", ".join(
            [
                f"target.`{column}` = source.`{column}`"
                for column in updated_bronze_df.columns
                if column != "insert_timestamp"
            ]
        )

        insert_columns = ", ".join([f"`{column}`" for column in updated_bronze_df.columns])
        insert_values = ", ".join([f"source.`{column}`" for column in updated_bronze_df.columns])

        print(f"  Merge condition: {merge_condition}")
        merge_result = spark.sql(f"""
                  MERGE INTO {target_table} AS target
                  USING {source_view} AS source
                  ON {merge_condition}
                  WHEN MATCHED THEN UPDATE SET {update_assignments}
                  WHEN NOT MATCHED THEN INSERT ({insert_columns}) VALUES ({insert_values})
                  """)
        merge_stats = merge_result.collect()[0].asDict()
        print(f"  MERGE load completed: {merge_stats}")
else:
    print(f"  ERROR: Unsupported load_type '{load_type}'")
    raise ValueError(f"Invalid load_type '{load_type}'. Must be one of 'FULL', 'APPEND', or 'MERGE'")

In [0]:
record_count = bronze_df.count()
target_count = spark.table(target_table).count()
print(f"Step 6: Load finished. Source records processed: {record_count}, total records in {target_table}: {target_count}")
dbutils.notebook.exit(str(record_count))